In [1]:
# ================= STORAGE ENGINE =================

class StorageEngine:

    def __init__(self):
        self.current_db = "default_db"
        self.transaction_active = False
        self.transaction_log = []

    def get_file(self, table):
        return self.current_db + "_" + table + ".txt"

    def get_meta_file(self):
        return self.current_db + "_tables.txt"

    # ---------- DATABASE ----------
    def use_database(self, db):
        print(f"\n📂 Switching Database → {db}")
        self.current_db = db
        print("✔ Database selected\n")

    # ---------- CREATE TABLE ----------
    def create_table(self, table):
        meta = self.get_meta_file()

        try:
            f = open(meta, "r")
            tables = f.read().splitlines()
            f.close()
        except:
            tables = []

        if table in tables:
            print(f"⚠ Table '{table}' already exists\n")
            return

        open(self.get_file(table), "w").close()

        f = open(meta, "a")
        f.write(table + "\n")
        f.close()

        print(f"✔ Table '{table}' created\n")

    # ---------- SHOW TABLES ----------
    def show_tables(self):
        meta = self.get_meta_file()

        print("\n📁 Tables:")

        try:
            f = open(meta, "r")
            tables = f.read().splitlines()
            f.close()

            if not tables:
                print("(no tables)")
            else:
                for t in tables:
                    print("→", t)

        except:
            print("(no tables)")

        print()

    # ---------- DROP TABLE ----------
    def drop(self, table):
        meta = self.get_meta_file()

        try:
            f = open(meta, "r")
            tables = f.read().splitlines()
            f.close()

            if table not in tables:
                print(f"❌ Table '{table}' not found\n")
                return

            tables.remove(table)

            f = open(meta, "w")
            for t in tables:
                f.write(t + "\n")
            f.close()

            # Clear file instead of deleting
            open(self.get_file(table), "w").close()

            print(f"🗑 Table '{table}' deleted\n")

        except:
            print(f"❌ Error deleting table\n")

    # ---------- INSERT ----------
    def insert(self, table, values):
        row = ",".join(values)

        if self.transaction_active:
            self.transaction_log.append(("INSERT", table, row))
            print("🧠 Insert saved in transaction\n")
        else:
            f = open(self.get_file(table), "a")
            f.write(row + "\n")
            f.close()
            print(f"✔ Record inserted into '{table}'\n")

    # ---------- SELECT ----------
    def select(self, table, condition=None):
        try:
            f = open(self.get_file(table), "r")
            data = f.readlines()
            f.close()

            print(f"\n📊 TABLE: {table}")
            print("-" * 30)

            if not data:
                print("(empty)")
            else:
                for row in data:
                    row = row.strip()
                    if condition:
                        if condition in row:
                            print("→", row)
                    else:
                        print("→", row)

            print("-" * 30 + "\n")

        except:
            print(f"❌ Table '{table}' not found\n")

    # ---------- DELETE ----------
    def delete(self, table, condition=None):
        try:
            f = open(self.get_file(table), "r")
            data = f.readlines()
            f.close()

            new_data = []

            for row in data:
                if condition:
                    if condition not in row:
                        new_data.append(row)
                else:
                    continue

            if self.transaction_active:
                self.transaction_log.append(("DELETE", table, new_data))
                print("🧠 Delete saved in transaction\n")
            else:
                f = open(self.get_file(table), "w")
                f.writelines(new_data)
                f.close()
                print(f"✔ Records deleted from '{table}'\n")

        except:
            print(f"❌ Table '{table}' not found\n")

    # ---------- UPDATE ----------
    def update(self, table, old, new):
        try:
            f = open(self.get_file(table), "r")
            data = f.readlines()
            f.close()

            updated = []

            for row in data:
                if old in row:
                    row = row.replace(old, new)
                updated.append(row)

            if self.transaction_active:
                self.transaction_log.append(("UPDATE", table, updated))
                print("🧠 Update saved in transaction\n")
            else:
                f = open(self.get_file(table), "w")
                f.writelines(updated)
                f.close()
                print(f"✔ Records updated in '{table}'\n")

        except:
            print(f"❌ Table '{table}' not found\n")

    # ---------- TRANSACTIONS ----------
    def begin(self):
        self.transaction_active = True
        self.transaction_log = []
        print("🚀 Transaction started\n")

    def commit(self):
        print("💾 Committing changes...\n")

        for action in self.transaction_log:
            if action[0] == "INSERT":
                f = open(self.get_file(action[1]), "a")
                f.write(action[2] + "\n")
                f.close()

            elif action[0] == "DELETE":
                f = open(self.get_file(action[1]), "w")
                f.writelines(action[2])
                f.close()

            elif action[0] == "UPDATE":
                f = open(self.get_file(action[1]), "w")
                f.writelines(action[2])
                f.close()

        self.transaction_active = False
        self.transaction_log = []
        print("✔ Transaction committed\n")

    def rollback(self):
        self.transaction_active = False
        self.transaction_log = []
        print("↩ Transaction rolled back\n")


# ================= PARSER =================

class Parser:

    def parse(self, query):

        query = query.strip().rstrip(";")

        if not query:
            return ("ERROR", "Empty Query")

        if query.upper().startswith("USE"):
            return ("USE", query.split()[1])

        elif query.upper().startswith("CREATE"):
            return ("CREATE", query.split()[2])

        elif query.upper().startswith("INSERT"):
            parts = query.split("VALUES")
            table = parts[0].split()[2]
            values = parts[1].replace("(", "").replace(")", "").split(",")
            return ("INSERT", table, [v.strip() for v in values])

        elif query.upper().startswith("SELECT"):
            tokens = query.split()
            if "WHERE" in query.upper():
                return ("SELECT_WHERE", tokens[-3], tokens[-1])
            return ("SELECT", tokens[-1])

        elif query.upper().startswith("DELETE"):
            tokens = query.split()
            if "WHERE" in query.upper():
                return ("DELETE_WHERE", tokens[2], tokens[-1])
            return ("DELETE", tokens[2])

        elif query.upper().startswith("UPDATE"):
            tokens = query.split()
            return ("UPDATE", tokens[1], tokens[-3], tokens[-1])

        elif query.upper().startswith("DROP"):
            return ("DROP", query.split()[2])

        elif query.upper() == "SHOW TABLES":
            return ("SHOW",)

        elif query.upper() == "BEGIN":
            return ("BEGIN",)

        elif query.upper() == "COMMIT":
            return ("COMMIT",)

        elif query.upper() == "ROLLBACK":
            return ("ROLLBACK",)

        else:
            return ("ERROR", "Invalid Query")


# ================= EXECUTOR =================

class Executor:

    def __init__(self):
        self.storage = StorageEngine()

    def execute(self, q):

        if q[0] == "USE":
            self.storage.use_database(q[1])

        elif q[0] == "CREATE":
            self.storage.create_table(q[1])

        elif q[0] == "INSERT":
            self.storage.insert(q[1], q[2])

        elif q[0] == "SELECT":
            self.storage.select(q[1])

        elif q[0] == "SELECT_WHERE":
            self.storage.select(q[1], q[2])

        elif q[0] == "DELETE":
            self.storage.delete(q[1])

        elif q[0] == "DELETE_WHERE":
            self.storage.delete(q[1], q[2])

        elif q[0] == "UPDATE":
            self.storage.update(q[1], q[2], q[3])

        elif q[0] == "DROP":
            self.storage.drop(q[1])

        elif q[0] == "SHOW":
            self.storage.show_tables()

        elif q[0] == "BEGIN":
            self.storage.begin()

        elif q[0] == "COMMIT":
            self.storage.commit()

        elif q[0] == "ROLLBACK":
            self.storage.rollback()

        else:
            print("❌ Error:", q[1], "\n")


# ================= CLI =================

class CLI:

    def start(self):

        parser = Parser()
        executor = Executor()

        print("=" * 45)
        print("        🚀 BuildDB Engine")
        print("=" * 45)
        print("Type queries ending with ';'")
        print("Type EXIT to quit\n")

        while True:

            query = input("BuildDB > ")

            if query.upper() == "EXIT":
                print("\n👋 Exiting BuildDB...")
                break

            parsed = parser.parse(query)
            executor.execute(parsed)


# ================= MAIN =================

if __name__ == "__main__":
    CLI().start()

        🚀 BuildDB Engine
Type queries ending with ';'
Type EXIT to quit



BuildDB >  show tables



📁 Tables:
(no tables)



BuildDB >  create table m


✔ Table 'm' created



...

❌ Error: Invalid Query 



BuildDB >  show tables



📁 Tables:
(no tables)



BuildDB >  exit



👋 Exiting BuildDB...
